# Polymer Combined Supervisor

Canonical unified polymer notebook. Edit the config cell below for one-off runs, or change `systems/polymer/notebook_params.py` for repo-wide defaults.

## User Config

In [ ]:
from pathlib import Path
import os

from systems.polymer import get_polymer_notebook_defaults
from systems.polymer.data_io import canonical_baseline_path
from utils.notebook_setup import prepare_polymer_notebook_env, print_grouped_notebook_summary

NB = get_polymer_notebook_defaults("combined")
RUN_MODE = NB["run_mode"]
STYLE_PROFILE = NB["style_profile"]
SAVE_PDF = NB["save_pdf"]
ENABLE_HORIZON = NB["enable_horizon"]
HORIZON_STATE_MODE = NB["horizon_state_mode"]
ENABLE_MATRIX = NB["enable_matrix"]
MATRIX_AGENT_KIND = NB["matrix_agent_kind"]
MATRIX_STATE_MODE = NB["matrix_state_mode"]
ENABLE_WEIGHTS = NB["enable_weights"]
WEIGHTS_AGENT_KIND = NB["weights_agent_kind"]
WEIGHTS_STATE_MODE = NB["weights_state_mode"]
ENABLE_RESIDUAL = NB["enable_residual"]
RESIDUAL_AGENT_KIND = NB["residual_agent_kind"]
RESIDUAL_STATE_MODE = NB["residual_state_mode"]
USE_RHO_AUTHORITY = NB["use_rho_authority"]
POLYMER_DATA_DIR_OVERRIDE = NB["data_dir_override"]
POLYMER_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]
RESULT_PREFIX_OVERRIDE = NB["result_prefix_override"]
COMPARE_PREFIX_OVERRIDE = NB["compare_prefix_override"]
BASELINE_MPC_PATH_OVERRIDE = NB["baseline_mpc_path_override"]
N_TESTS_OVERRIDE = NB["n_tests_override"]
SET_POINTS_LEN_OVERRIDE = NB["set_points_len_override"]
WARM_START_OVERRIDE = NB["warm_start_override"]
TEST_CYCLE_OVERRIDE = NB["test_cycle_override"]
PLOT_START_EPISODE_OVERRIDE = NB["plot_start_episode_override"]
COMPARE_START_EPISODE_OVERRIDE = NB["compare_start_episode_override"]
REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_polymer_notebook_env(data_dir_override=POLYMER_DATA_DIR_OVERRIDE, results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE)
os.chdir(REPO_ROOT)
RUN_PROFILE = NB["run_profiles"][RUN_MODE]

def build_combined_suffix() -> str:
    parts = []
    if ENABLE_HORIZON:
        parts.append(f"h_dqn_{HORIZON_STATE_MODE}")
    if ENABLE_MATRIX:
        parts.append(f"m_{MATRIX_AGENT_KIND}_{MATRIX_STATE_MODE}")
    if ENABLE_WEIGHTS:
        parts.append(f"w_{WEIGHTS_AGENT_KIND}_{WEIGHTS_STATE_MODE}")
    if ENABLE_RESIDUAL:
        rho_suffix = "rho" if USE_RHO_AUTHORITY else "no_rho"
        parts.append(f"r_{RESIDUAL_AGENT_KIND}_{RESIDUAL_STATE_MODE}_{rho_suffix}")
    return "__".join(parts) if parts else "baseline"


## Imports

In [ ]:
import numpy as np
import torch

from DQN.dqn_agent import DQNAgent
from SACAgent.sac_agent import SACAgent
from Simulation.mpc import MpcSolverGeneral
from Simulation.system_functions import PolymerCSTR
from TD3Agent.agent import TD3Agent
from systems.polymer import (
    HORIZON_CONTROL_GRID,
    HORIZON_PREDICT_GRID,
    POLYMER_DELTA_T_HOURS,
    POLYMER_DESIGN_PARAMS,
    POLYMER_INPUT_BOUNDS,
    POLYMER_OBSERVER_POLES,
    POLYMER_RL_SETPOINTS_PHYS,
    POLYMER_SETPOINT_RANGE_PHYS,
    POLYMER_SYSTEM_METADATA,
    POLYMER_SS_INPUTS,
    POLYMER_SYSTEM_PARAMS,
    RL_REWARD_DEFAULTS,
    load_polymer_system_data,
)
from utils.combined_runner import run_combined_supervisor
from utils.helpers import apply_min_max, build_horizon_recipes
from utils.observer import compute_observer_gain
from utils.plotting import plot_combined_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim, default_mismatch_scale


## System And Data Setup

In [ ]:
# Build the plant, load the canonical data bundle, and prepare the supervisory setpoint scenario.
SYS = NB["system_setup"]
system_params = SYS["system_params"].copy()
system_design_params = SYS["design_params"].copy()
system_steady_state_inputs = SYS["ss_inputs"].copy()
delta_t = SYS["delta_t_hours"]
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr_ss.ss_inputs, "y_ss": cstr_ss.y_ss}
setpoint_y = SYS["setpoint_range_phys"].copy()
u_min = SYS["input_bounds"]["u_min"].copy()
u_max = SYS["input_bounds"]["u_max"].copy()
system_data = load_polymer_system_data(REPO_ROOT, steady_states=steady_states, setpoint_y=setpoint_y, u_min=u_min, u_max=u_max, n_inputs=2, data_override=POLYMER_DATA_DIR_OVERRIDE)
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario_phys = SYS["rl_setpoints_phys"].copy()
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])


## Run / Reward / Agent Setup

In [ ]:
# Run-profile, controller, reward, and agent setup.
EPISODE_CFG = NB["episode_defaults"]
CTRL = NB["controller"]
HORIZON_CFG = NB["horizon_agent"]
TD3_CFG = NB["td3_agent"]
SAC_CFG = NB["sac_agent"]
REWARD_CFG = NB["reward"]

n_tests = int(EPISODE_CFG["n_tests"] if N_TESTS_OVERRIDE is None else N_TESTS_OVERRIDE)
set_points_len = int(EPISODE_CFG["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else SET_POINTS_LEN_OVERRIDE)
warm_start = int(EPISODE_CFG["warm_start"] if WARM_START_OVERRIDE is None else WARM_START_OVERRIDE)
TEST_CYCLE = list(EPISODE_CFG["test_cycle"] if TEST_CYCLE_OVERRIDE is None else TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = int(RUN_PROFILE["plot_start_episode"] if PLOT_START_EPISODE_OVERRIDE is None else PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = int(RUN_PROFILE["compare_start_episode"] if COMPARE_START_EPISODE_OVERRIDE is None else COMPARE_START_EPISODE_OVERRIDE)
ACTIVE_AGENT_SUFFIX = build_combined_suffix()
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or RUN_PROFILE["result_prefix_template"].format(suffix=ACTIVE_AGENT_SUFFIX)
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or RUN_PROFILE["compare_prefix_template"].format(suffix=ACTIVE_AGENT_SUFFIX)
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else canonical_baseline_path(REPO_ROOT, RUN_MODE, data_override=POLYMER_DATA_DIR_OVERRIDE)
N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
MISMATCH_SCALE = default_mismatch_scale(min_max_dict)
MISMATCH_CLIP = CTRL["mismatch_clip"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HORIZON_EXPLORATION_MODE = HORIZON_CFG["exploration_mode"]
HORIZON_LOSS_TYPE = HORIZON_CFG["loss_type"]
HORIZON_BUFFER_SIZE = int(HORIZON_CFG["buffer_size"])
HORIZON_REPLAY_FRAC_PER = float(HORIZON_CFG["replay_frac_per"])
HORIZON_REPLAY_FRAC_RECENT = float(HORIZON_CFG["replay_frac_recent"])
HORIZON_REPLAY_RECENT_WINDOW_MULT = int(HORIZON_CFG["replay_recent_window_mult"])
HORIZON_REPLAY_RECENT_WINDOW = int(HORIZON_CFG["replay_recent_window"]) if HORIZON_CFG["replay_recent_window"] is not None else min(HORIZON_BUFFER_SIZE, HORIZON_REPLAY_RECENT_WINDOW_MULT * set_points_len)
HORIZON_REPLAY_ALPHA = float(HORIZON_CFG["replay_alpha"])
HORIZON_REPLAY_BETA_START = float(HORIZON_CFG["replay_beta_start"])
HORIZON_REPLAY_BETA_END = float(HORIZON_CFG["replay_beta_end"])
HORIZON_REPLAY_BETA_STEPS = int(HORIZON_CFG["replay_beta_steps"])
TD3_EXPLORATION_MODE = TD3_CFG["exploration_mode"]
TD3_LOSS_TYPE = TD3_CFG["loss_type"]
SAC_LOSS_TYPE = SAC_CFG["loss_type"]
USE_SHIFTED_MPC_WARM_START = CTRL["use_shifted_mpc_warm_start"]
INPUT_TIGHTENING_FRAC = float(CTRL["input_tightening_frac"])
ENABLE_ACCEPT_NORM_TEST = bool(CTRL["enable_accept_norm_test"])
EPS_A_NORM_FRAC = float(CTRL["eps_A_norm_frac"])
EPS_B_NORM_FRAC = float(CTRL["eps_B_norm_frac"])
ENABLE_ACCEPT_PREDICTION_TEST = bool(CTRL["enable_accept_prediction_test"])
PREDICTION_CHECK_HORIZON = int(CTRL["prediction_check_horizon"])
EPS_Y_PRED_SCALED = float(CTRL["eps_y_pred_scaled"])
ENABLE_SOLVER_FALLBACK = bool(CTRL["enable_solver_fallback"])
PROBE_INPUT_MODE = CTRL["probe_input_mode"]
DECISION_INTERVAL = int(CTRL["decision_interval"])
TD3_PARAM_NOISE_RESAMPLE_INTERVAL = TD3_CFG["param_noise_resample_interval"]
TD3_BUFFER_SIZE = int(TD3_CFG["buffer_size"])
TD3_REPLAY_FRAC_PER = float(TD3_CFG["replay_frac_per"])
TD3_REPLAY_FRAC_RECENT = float(TD3_CFG["replay_frac_recent"])
TD3_REPLAY_RECENT_WINDOW_MULT = int(TD3_CFG["replay_recent_window_mult"])
TD3_REPLAY_RECENT_WINDOW = int(TD3_CFG["replay_recent_window"]) if TD3_CFG["replay_recent_window"] is not None else min(TD3_BUFFER_SIZE, TD3_REPLAY_RECENT_WINDOW_MULT * set_points_len)
TD3_REPLAY_ALPHA = float(TD3_CFG["replay_alpha"])
TD3_REPLAY_BETA_START = float(TD3_CFG["replay_beta_start"])
TD3_REPLAY_BETA_END = float(TD3_CFG["replay_beta_end"])
TD3_REPLAY_BETA_STEPS = int(TD3_CFG["replay_beta_steps"])
SAC_BUFFER_SIZE = int(SAC_CFG["buffer_size"])
SAC_REPLAY_FRAC_PER = float(SAC_CFG["replay_frac_per"])
SAC_REPLAY_FRAC_RECENT = float(SAC_CFG["replay_frac_recent"])
SAC_REPLAY_RECENT_WINDOW_MULT = int(SAC_CFG["replay_recent_window_mult"])
SAC_REPLAY_RECENT_WINDOW = int(SAC_CFG["replay_recent_window"]) if SAC_CFG["replay_recent_window"] is not None else min(SAC_BUFFER_SIZE, SAC_REPLAY_RECENT_WINDOW_MULT * set_points_len)
SAC_REPLAY_ALPHA = float(SAC_CFG["replay_alpha"])
SAC_REPLAY_BETA_START = float(SAC_CFG["replay_beta_start"])
SAC_REPLAY_BETA_END = float(SAC_CFG["replay_beta_end"])
SAC_REPLAY_BETA_STEPS = int(SAC_CFG["replay_beta_steps"])
PREDICT_GRID = list(CTRL["predict_grid"])
CONTROL_GRID = list(CTRL["control_grid"])
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
predict_h = CTRL["predict_h"]
cont_h = CTRL["cont_h"]
Q1_penalty = CTRL["Q1_penalty"]
Q2_penalty = CTRL["Q2_penalty"]
R1_penalty = CTRL["R1_penalty"]
R2_penalty = CTRL["R2_penalty"]
MODEL_LOW = CTRL["model_low"].copy()
MODEL_HIGH = CTRL["model_high"].copy()
WEIGHTS_LOW = CTRL["weights_low"].copy()
WEIGHTS_HIGH = CTRL["weights_high"].copy()
RESIDUAL_LOW = CTRL["residual_low"].copy()
RESIDUAL_HIGH = CTRL["residual_high"].copy()
nominal_qs = CTRL["nominal_qs"]
nominal_qi = CTRL["nominal_qi"]
nominal_hA = CTRL["nominal_ha"]
qi_change = CTRL["qi_change"]
qs_change = CTRL["qs_change"]
ha_change = CTRL["ha_change"]
REPLAY_SETTINGS = {
    "horizon": {
        "buffer_size": HORIZON_BUFFER_SIZE,
        "replay_frac_per": HORIZON_REPLAY_FRAC_PER,
        "replay_frac_recent": HORIZON_REPLAY_FRAC_RECENT,
        "replay_recent_window_mult": HORIZON_REPLAY_RECENT_WINDOW_MULT,
        "replay_recent_window": HORIZON_REPLAY_RECENT_WINDOW,
        "replay_alpha": HORIZON_REPLAY_ALPHA,
        "replay_beta_start": HORIZON_REPLAY_BETA_START,
        "replay_beta_end": HORIZON_REPLAY_BETA_END,
        "replay_beta_steps": HORIZON_REPLAY_BETA_STEPS,
    },
    "td3": {
        "buffer_size": TD3_BUFFER_SIZE,
        "replay_frac_per": TD3_REPLAY_FRAC_PER,
        "replay_frac_recent": TD3_REPLAY_FRAC_RECENT,
        "replay_recent_window_mult": TD3_REPLAY_RECENT_WINDOW_MULT,
        "replay_recent_window": TD3_REPLAY_RECENT_WINDOW,
        "replay_alpha": TD3_REPLAY_ALPHA,
        "replay_beta_start": TD3_REPLAY_BETA_START,
        "replay_beta_end": TD3_REPLAY_BETA_END,
        "replay_beta_steps": TD3_REPLAY_BETA_STEPS,
    },
    "sac": {
        "buffer_size": SAC_BUFFER_SIZE,
        "replay_frac_per": SAC_REPLAY_FRAC_PER,
        "replay_frac_recent": SAC_REPLAY_FRAC_RECENT,
        "replay_recent_window_mult": SAC_REPLAY_RECENT_WINDOW_MULT,
        "replay_recent_window": SAC_REPLAY_RECENT_WINDOW,
        "replay_alpha": SAC_REPLAY_ALPHA,
        "replay_beta_start": SAC_REPLAY_BETA_START,
        "replay_beta_end": SAC_REPLAY_BETA_END,
        "replay_beta_steps": SAC_REPLAY_BETA_STEPS,
    },
}
MPC_obj = MpcSolverGeneral(A_aug, B_aug, C_aug, Q_out=np.array([Q1_penalty, Q2_penalty], float), R_in=np.array([R1_penalty, R2_penalty], float), NP=predict_h, NC=cont_h)
# Reward setup.
reward_params, reward_fn = make_reward_fn_relative_QR(data_min, data_max, N_INPUTS, **REWARD_CFG)

def make_continuous_agent(agent_kind, state_dim, action_dim):
    if agent_kind == "td3":
        return TD3Agent(state_dim=state_dim, action_dim=action_dim, actor_hidden=list(TD3_CFG["actor_hidden"]), critic_hidden=list(TD3_CFG["critic_hidden"]), gamma=TD3_CFG["gamma"], actor_lr=TD3_CFG["actor_lr"], critic_lr=TD3_CFG["critic_lr"], batch_size=TD3_CFG["batch_size"], policy_delay=TD3_CFG["policy_delay"], target_policy_smoothing_noise_std=TD3_CFG["target_policy_smoothing_noise_std"], noise_clip=TD3_CFG["noise_clip"], max_action=TD3_CFG["max_action"], tau=TD3_CFG["tau"], std_start=TD3_CFG["std_start"], std_end=TD3_CFG["std_end"], std_decay_rate=TD3_CFG["std_decay_rate"], std_decay_mode=TD3_CFG["std_decay_mode"], buffer_size=TD3_BUFFER_SIZE, replay_frac_per=TD3_REPLAY_FRAC_PER, replay_frac_recent=TD3_REPLAY_FRAC_RECENT, replay_recent_window=TD3_REPLAY_RECENT_WINDOW, replay_alpha=TD3_REPLAY_ALPHA, replay_beta_start=TD3_REPLAY_BETA_START, replay_beta_end=TD3_REPLAY_BETA_END, replay_beta_steps=TD3_REPLAY_BETA_STEPS, device=DEVICE, actor_freeze=TD3_CFG["actor_freeze"], exploration_mode=TD3_EXPLORATION_MODE, loss_type=TD3_LOSS_TYPE, param_noise_resample_interval=TD3_PARAM_NOISE_RESAMPLE_INTERVAL)
    if agent_kind == "sac":
        target_entropy = -action_dim if SAC_CFG["target_entropy"] == "auto_negative_action_dim" else SAC_CFG["target_entropy"]
        return SACAgent(state_dim=state_dim, action_dim=action_dim, actor_hidden=list(SAC_CFG["actor_hidden"]), critic_hidden=list(SAC_CFG["critic_hidden"]), gamma=SAC_CFG["gamma"], actor_lr=SAC_CFG["actor_lr"], critic_lr=SAC_CFG["critic_lr"], alpha_lr=SAC_CFG["alpha_lr"], batch_size=SAC_CFG["batch_size"], grad_clip_norm=SAC_CFG["grad_clip_norm"], init_alpha=SAC_CFG["init_alpha"], learn_alpha=SAC_CFG["learn_alpha"], target_entropy=target_entropy, target_update=SAC_CFG["target_update"], tau=SAC_CFG["tau"], hard_update_interval=SAC_CFG["hard_update_interval"], activation=SAC_CFG["activation"], use_layernorm=SAC_CFG["use_layernorm"], dropout=SAC_CFG["dropout"], max_action=SAC_CFG["max_action"], buffer_size=SAC_BUFFER_SIZE, replay_frac_per=SAC_REPLAY_FRAC_PER, replay_frac_recent=SAC_REPLAY_FRAC_RECENT, replay_recent_window=SAC_REPLAY_RECENT_WINDOW, replay_alpha=SAC_REPLAY_ALPHA, replay_beta_start=SAC_REPLAY_BETA_START, replay_beta_end=SAC_REPLAY_BETA_END, replay_beta_steps=SAC_REPLAY_BETA_STEPS, device=DEVICE, use_adamw=SAC_CFG["use_adamw"], actor_freeze=SAC_CFG["actor_freeze"], loss_type=SAC_LOSS_TYPE)
    raise ValueError("Continuous agent kind must be 'td3' or 'sac'.")

horizon_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, HORIZON_STATE_MODE)
matrix_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, MATRIX_STATE_MODE)
weights_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, WEIGHTS_STATE_MODE)
residual_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, RESIDUAL_STATE_MODE)

# Agent setup.
horizon_agent = DQNAgent(state_dim=horizon_state_dim, action_dim=len(HORIZON_RECIPES), hidden_dim=list(HORIZON_CFG["hidden_layers"]), gamma=HORIZON_CFG["gamma"], lr=HORIZON_CFG["lr"], batch_size=HORIZON_CFG["batch_size"], buffer_size=HORIZON_BUFFER_SIZE, replay_frac_per=HORIZON_REPLAY_FRAC_PER, replay_frac_recent=HORIZON_REPLAY_FRAC_RECENT, replay_recent_window=HORIZON_REPLAY_RECENT_WINDOW, replay_alpha=HORIZON_REPLAY_ALPHA, replay_beta_start=HORIZON_REPLAY_BETA_START, replay_beta_end=HORIZON_REPLAY_BETA_END, replay_beta_steps=HORIZON_REPLAY_BETA_STEPS, grad_clip_norm=HORIZON_CFG["grad_clip_norm"], double_dqn=HORIZON_CFG["double_dqn"], target_update=HORIZON_CFG["target_update"], tau=HORIZON_CFG["tau"], hard_update_interval=HORIZON_CFG["hard_update_interval"], activation=HORIZON_CFG["activation"], use_layer_norm=HORIZON_CFG["use_layer_norm"], dropout=HORIZON_CFG["dropout"], device=DEVICE, exploration_mode=HORIZON_EXPLORATION_MODE, loss_type=HORIZON_LOSS_TYPE, eps_start=HORIZON_CFG["eps_start"], eps_end=HORIZON_CFG["eps_end"], eps_decay_rate=HORIZON_CFG["eps_decay_rate"], eps_decay_mode=HORIZON_CFG["eps_decay_mode"], target_combine=HORIZON_CFG["target_combine"])
matrix_agent = make_continuous_agent(MATRIX_AGENT_KIND, matrix_state_dim, 1 + N_INPUTS)
weights_agent = make_continuous_agent(WEIGHTS_AGENT_KIND, weights_state_dim, 4)
residual_agent = make_continuous_agent(RESIDUAL_AGENT_KIND, residual_state_dim, N_INPUTS)


## Resolved Summary

In [ ]:
print_grouped_notebook_summary(
    "Polymer combined run summary",
    {
        "Paths": {"Repo root": REPO_ROOT, "Data dir": DATA_DIR, "Results dir": RESULT_DIR, "Baseline MPC": BASELINE_MPC_PATH},
        "Run setup": {"Run mode": RUN_MODE, "n_tests": n_tests, "set_points_len": set_points_len, "warm_start": warm_start, "test_cycle": TEST_CYCLE, "decision_interval": DECISION_INTERVAL, "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START, "matrix_estimator_mode": "fixed_nominal"},
        "System / controller": {"predict_h": predict_h, "cont_h": cont_h, "observer_poles": POLYMER_OBSERVER_POLES.tolist(), "predict_grid": PREDICT_GRID, "control_grid": CONTROL_GRID},
        "Matrix robust prediction": {"input_tightening_frac": INPUT_TIGHTENING_FRAC, "enable_accept_norm_test": ENABLE_ACCEPT_NORM_TEST, "eps_A_norm_frac": EPS_A_NORM_FRAC, "eps_B_norm_frac": EPS_B_NORM_FRAC, "enable_accept_prediction_test": ENABLE_ACCEPT_PREDICTION_TEST, "prediction_check_horizon": PREDICTION_CHECK_HORIZON, "eps_y_pred_scaled": EPS_Y_PRED_SCALED, "enable_solver_fallback": ENABLE_SOLVER_FALLBACK, "probe_input_mode": PROBE_INPUT_MODE},
        "Reward": reward_params,
        "Agents": {"horizon": f"enabled={ENABLE_HORIZON}, state_mode={HORIZON_STATE_MODE}", "matrix": f"enabled={ENABLE_MATRIX}, kind={MATRIX_AGENT_KIND}, state_mode={MATRIX_STATE_MODE}", "weights": f"enabled={ENABLE_WEIGHTS}, kind={WEIGHTS_AGENT_KIND}, state_mode={WEIGHTS_STATE_MODE}", "residual": f"enabled={ENABLE_RESIDUAL}, kind={RESIDUAL_AGENT_KIND}, state_mode={RESIDUAL_STATE_MODE}, rho={USE_RHO_AUTHORITY}"},
        "Replay": REPLAY_SETTINGS,
        "Plotting / export": {"style_profile": STYLE_PROFILE, "save_pdf": SAVE_PDF, "result_prefix": RESULT_PREFIX, "compare_prefix": COMPARE_PREFIX, "plot_start_episode": PLOT_START_EPISODE, "compare_start_episode": COMPARE_START_EPISODE},
    },
)

## Run

In [ ]:
# Assemble the shared runner configuration and execute the rollout.
combined_cfg = {
    "run_mode": RUN_MODE,
    "notebook_source": "RL_assisted_MPC_combined_unified.ipynb",
    "decision_interval": DECISION_INTERVAL,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "horizon_cfg": {
        "enabled": ENABLE_HORIZON,
        "state_mode": HORIZON_STATE_MODE,
        "mismatch_scale": MISMATCH_SCALE,
        "mismatch_clip": MISMATCH_CLIP,
        "horizon_recipes": HORIZON_RECIPES,
        "default_horizons": (predict_h, cont_h),
    },
    "matrix_cfg": {
        "enabled": ENABLE_MATRIX,
        "agent_kind": MATRIX_AGENT_KIND,
        "state_mode": MATRIX_STATE_MODE,
        "mismatch_scale": MISMATCH_SCALE,
        "mismatch_clip": MISMATCH_CLIP,
        "low_coef": MODEL_LOW,
        "high_coef": MODEL_HIGH,
        "input_tightening_frac": INPUT_TIGHTENING_FRAC,
        "enable_accept_norm_test": ENABLE_ACCEPT_NORM_TEST,
        "eps_A_norm_frac": EPS_A_NORM_FRAC,
        "eps_B_norm_frac": EPS_B_NORM_FRAC,
        "enable_accept_prediction_test": ENABLE_ACCEPT_PREDICTION_TEST,
        "prediction_check_horizon": PREDICTION_CHECK_HORIZON,
        "eps_y_pred_scaled": EPS_Y_PRED_SCALED,
        "enable_solver_fallback": ENABLE_SOLVER_FALLBACK,
        "probe_input_mode": PROBE_INPUT_MODE,
    },
    "weight_cfg": {
        "enabled": ENABLE_WEIGHTS,
        "agent_kind": WEIGHTS_AGENT_KIND,
        "state_mode": WEIGHTS_STATE_MODE,
        "mismatch_scale": MISMATCH_SCALE,
        "mismatch_clip": MISMATCH_CLIP,
        "low_coef": WEIGHTS_LOW,
        "high_coef": WEIGHTS_HIGH,
    },
    "residual_cfg": {
        "enabled": ENABLE_RESIDUAL,
        "agent_kind": RESIDUAL_AGENT_KIND,
        "state_mode": RESIDUAL_STATE_MODE,
        "mismatch_scale": MISMATCH_SCALE,
        "mismatch_clip": MISMATCH_CLIP,
        "use_rho_authority": USE_RHO_AUTHORITY,
        "low_coef": RESIDUAL_LOW,
        "high_coef": RESIDUAL_HIGH,
    },
}

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
agents = {}
if ENABLE_HORIZON:
    agents["horizon"] = horizon_agent
if ENABLE_MATRIX:
    agents["matrix"] = matrix_agent
if ENABLE_WEIGHTS:
    agents["weights"] = weights_agent
if ENABLE_RESIDUAL:
    agents["residual"] = residual_agent

runtime_ctx = {
    "system": cstr,
    "agents": agents,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": POLYMER_OBSERVER_POLES.copy(),
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": POLYMER_SYSTEM_METADATA,
    "reward_params": reward_params,
}

result_bundle = run_combined_supervisor(combined_cfg=combined_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH


## Plotting And Export

In [ ]:
# Generate saved figures and any requested MPC comparison plots.
out_dir_rl = plot_combined_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
        "reward_fn": reward_fn,
        "system_metadata": POLYMER_SYSTEM_METADATA,
        "include_baseline_compare": True,
        "compare_mode": RUN_PROFILE["compare_mode"],
        "compare_prefix": COMPARE_PREFIX,
        "compare_directory": RESULT_DIR,
        "mpc_path_or_dir": BASELINE_MPC_PATH,
        "compare_start_episode": COMPARE_START_EPISODE,
    },
)

print("Combined result directory:", out_dir_rl)
